# Messy Mashup - Inference Notebook
## Kaggle Submission (Internet OFF, GPU ON)
Loads pretrained models and generates submission.csv

In [ ]:
# ============================================================
# Cell 1: Imports (no pip installs needed - all pre-installed)
# ============================================================
import os, random, warnings, numpy as np, pandas as pd
import librosa, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from pathlib import Path
from tqdm.auto import tqdm
import joblib
from transformers import ASTForAudioClassification, ASTFeatureExtractor

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# ============================================================
# Cell 2: Paths & Config
# ============================================================
DATA_PATH  = Path('/kaggle/input/messy-mashup')
MODEL_PATH = Path('/kaggle/input/messy-mashup-models')   # your uploaded model dataset
AST_PATH   = Path('/kaggle/input/ast-pretrained')         # your uploaded HF cache

GENRES = ["blues", "classical", "country", "disco", "hiphop",
          "jazz", "metal", "pop", "reggae", "rock"]
IDX2GENRE = {i: g for i, g in enumerate(GENRES)}

SR = 22050
DURATION = 10
N_MELS = 128
IMG_WIDTH = 216
N_FOLDS = 5

In [ ]:
# ============================================================
# Cell 3: Audio Utilities
# ============================================================
def load_audio(path, sr=SR, duration=DURATION):
    try:
        y, _ = librosa.load(path, sr=sr, duration=duration, mono=True)
    except Exception:
        y = np.zeros(sr * duration, dtype=np.float32)
    target_len = sr * duration
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]
    return y.astype(np.float32)

def audio_to_melspec(y, sr=SR):
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, n_fft=2048,
                                         hop_length=512, fmax=8000)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
    if mel_db.shape[1] < IMG_WIDTH:
        mel_db = np.pad(mel_db, ((0, 0), (0, IMG_WIDTH - mel_db.shape[1])))
    else:
        mel_db = mel_db[:, :IMG_WIDTH]
    return mel_db.astype(np.float32)

def extract_features(y, sr=SR):
    features = []
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    features.extend(mfcc.mean(axis=1)); features.extend(mfcc.std(axis=1))
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    features.extend(chroma.mean(axis=1)); features.extend(chroma.std(axis=1))
    features.append(librosa.feature.spectral_centroid(y=y, sr=sr).mean())
    features.append(librosa.feature.spectral_bandwidth(y=y, sr=sr).mean())
    features.append(librosa.feature.spectral_rolloff(y=y, sr=sr).mean())
    features.append(librosa.feature.spectral_contrast(y=y, sr=sr).mean())
    features.append(librosa.feature.tempo(y=y, sr=sr)[0])
    features.append(librosa.feature.zero_crossing_rate(y).mean())
    features.append(librosa.feature.rms(y=y).mean())
    return np.array(features, dtype=np.float32)

In [ ]:
# ============================================================
# Cell 4: Dataset Classes
# ============================================================
class MelDataset(Dataset):
    def __init__(self, specs):
        self.specs = specs
    def __len__(self):
        return len(self.specs)
    def __getitem__(self, idx):
        return torch.FloatTensor(self.specs[idx]).unsqueeze(0)

class ASTDataset(Dataset):
    def __init__(self, audios, extractor, sr=SR):
        self.audios = audios
        self.extractor = extractor
        self.sr = sr
    def __len__(self):
        return len(self.audios)
    def __getitem__(self, idx):
        inputs = self.extractor(self.audios[idx], sampling_rate=self.sr,
                                return_tensors="pt", padding="max_length",
                                max_length=1024, truncation=True)
        return inputs['input_values'].squeeze(0)

In [ ]:
# ============================================================
# Cell 5: Model Definitions
# ============================================================
class GenreClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.3),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.4), nn.Linear(256, 128), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x.view(x.size(0), -1))

class CRNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d((2, 2)), nn.Dropout(0.2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d((2, 2)), nn.Dropout(0.2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d((2, 1)), nn.Dropout(0.3),
        )
        self.freq_pool = nn.AdaptiveAvgPool2d((1, None))
        self.gru = nn.GRU(128, 128, num_layers=2, batch_first=True,
                          dropout=0.3, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Dropout(0.4), nn.Linear(256, 64), nn.ReLU(),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        x = self.cnn(x)
        x = self.freq_pool(x).squeeze(2).permute(0, 2, 1)
        out, _ = self.gru(x)
        return self.classifier(out[:, -1, :])

class ASTClassifier(nn.Module):
    def __init__(self, num_classes=10, model_path=None):
        super().__init__()
        self.ast = ASTForAudioClassification.from_pretrained(
            model_path, num_labels=num_classes, ignore_mismatched_sizes=True
        )
    def forward(self, x):
        return self.ast(x).logits

In [ ]:
# ============================================================
# Cell 6: Load Test Data
# ============================================================
test_df = pd.read_csv(DATA_PATH / 'test.csv')
mashups_path = DATA_PATH / 'mashups'

# Auto-detect filename pattern
if 'filename' in test_df.columns:
    test_files = [mashups_path / row['filename'] for _, row in test_df.iterrows()]
else:
    sample_id = test_df['id'].iloc[0]
    if (mashups_path / f"{sample_id}.wav").exists():
        test_files = [mashups_path / f"{row['id']}.wav" for _, row in test_df.iterrows()]
    elif (mashups_path / f"mashup_{sample_id}.wav").exists():
        test_files = [mashups_path / f"mashup_{row['id']}.wav" for _, row in test_df.iterrows()]
    else:
        all_wavs = sorted(mashups_path.glob('*.wav'))
        test_files = all_wavs[:len(test_df)]

print(f"Test files: {len(test_files)}")

# Load audio once
print("Loading test audio...")
test_audios = np.array([load_audio(str(f)) for f in tqdm(test_files)])

# Mel spectrograms
print("Computing mel spectrograms...")
test_specs = np.array([audio_to_melspec(y) for y in tqdm(test_audios)])

# Handcrafted features
print("Extracting features...")
X_test_feat = np.array([extract_features(y) for y in tqdm(test_audios)])
scaler = joblib.load(MODEL_PATH / 'scaler.pkl')
X_test_scaled = scaler.transform(X_test_feat)

print(f"Specs: {test_specs.shape} | Features: {X_test_feat.shape}")

In [ ]:
# ============================================================
# Cell 7: Load All Models
# ============================================================

@torch.no_grad()
def get_probs(models, loader, device):
    all_probs = []
    for model in models:
        model.eval()
        probs = []
        for batch in loader:
            x = batch.to(device)
            logits = model(x)
            probs.append(F.softmax(logits, dim=1).cpu().numpy())
        all_probs.append(np.concatenate(probs, axis=0))
    return np.mean(all_probs, axis=0)

# Shared mel loader
test_mel_loader = DataLoader(MelDataset(test_specs), batch_size=64, shuffle=False, num_workers=2)

# --- CNN ---
print("Loading CNN models...")
cnn_models = []
for i in range(N_FOLDS):
    m = GenreClassifier(10).to(device)
    m.load_state_dict(torch.load(MODEL_PATH / f'cnn_fold{i}.pth', map_location=device))
    m.eval()
    cnn_models.append(m)
cnn_probs = get_probs(cnn_models, test_mel_loader, device)
print(f"  CNN done: {cnn_probs.shape}")

# --- CRNN ---
print("Loading CRNN models...")
crnn_models = []
for i in range(N_FOLDS):
    m = CRNN(10).to(device)
    m.load_state_dict(torch.load(MODEL_PATH / f'crnn_fold{i}.pth', map_location=device))
    m.eval()
    crnn_models.append(m)
crnn_probs = get_probs(crnn_models, test_mel_loader, device)
print(f"  CRNN done: {crnn_probs.shape}")

# --- AST ---
print("Loading AST model...")
ast_extractor = ASTFeatureExtractor.from_pretrained(str(AST_PATH))
ast_model = ASTClassifier(10, model_path=str(AST_PATH)).to(device)
ast_model.load_state_dict(torch.load(MODEL_PATH / 'ast_best.pth', map_location=device))
ast_model.eval()
ast_test_loader = DataLoader(ASTDataset(test_audios, ast_extractor), batch_size=32,
                              shuffle=False, num_workers=2)
ast_probs = get_probs([ast_model], ast_test_loader, device)
print(f"  AST done: {ast_probs.shape}")

# --- XGBoost ---
print("Loading XGBoost model...")
xgb_model = joblib.load(MODEL_PATH / 'xgb_model.pkl')
xgb_probs = xgb_model.predict_proba(X_test_scaled)
print(f"  XGBoost done: {xgb_probs.shape}")

print("\nAll models loaded and predictions complete.")

In [ ]:
# ============================================================
# Cell 8: Weighted Ensemble & Submission
# ============================================================
W_CNN, W_CRNN, W_AST, W_XGB = 0.25, 0.20, 0.40, 0.15

final_probs = (W_CNN * cnn_probs + W_CRNN * crnn_probs +
               W_AST * ast_probs + W_XGB * xgb_probs)

final_preds = final_probs.argmax(axis=1)
predicted_genres = [IDX2GENRE[p] for p in final_preds]

submission = pd.DataFrame({'id': test_df['id'], 'genre': predicted_genres})

# Sanity checks
assert len(submission) == len(test_df)
assert set(submission['genre'].unique()).issubset(set(GENRES))

print(f"Predictions: {len(submission)}")
print(f"\nDistribution:\n{submission['genre'].value_counts().to_string()}")

submission.to_csv('submission.csv', index=False)
print(f"\nsubmission.csv saved.")
print(submission.head(10))